# Day 2 — Notebook 2: Build a ReAct Tax FAQ Agent
**Tax Tech Transformation Program | Big4 Firm**

## What you will build
A functional **Tax FAQ Agent** that:
- Holds a curated tax knowledge base
- Uses **tool calling** (search + calculator) — the ReAct pattern
- Maintains **conversation memory** across multiple turns
- Prints a visible **reasoning trace** so you can see _why_ it calls each tool

## Lab Structure
| Part | Activity | Time |
|------|----------|------|
| Part 1 | Re-run setup & imports | 5 min |
| Part 2 | Build the knowledge base + search tool | 30 min |
| Part 3 | Implement ReAct loop | 40 min |
| Part 4 | Test, iterate, extend | 30 min |

> **Prerequisite:** Complete Notebook 1 first so `client` and `AZURE_DEPLOYMENT_NAME` are defined.

## Part 1: Imports & Client Setup

In [1]:
# Run if you're starting fresh — skip if client is already defined
%pip install openai --quiet

import os, json, textwrap
from getpass import getpass
from openai import AzureOpenAI


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
AZURE_OPENAI_ENDPOINT  = input("Endpoint: ").strip()
AZURE_OPENAI_API_KEY   = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME  = input("Deployment name: ").strip()

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = "2024-02-01",
)
print("✅ Client ready")

Alternatively, you can uncomment and run the following section to use .env variables.

In [4]:
# from dotenv import load_dotenv

# load_dotenv()

# AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
# AZURE_DEPLOYMENT_NAME = os.getenv("AZURE_DEPLOYMENT_NAME")
# AZURE_API_VERSION = os.getenv("AZURE_API_VERSION")

# # Initialise client
# client = AzureOpenAI(
#     azure_endpoint = AZURE_OPENAI_ENDPOINT,
#     api_key        = AZURE_OPENAI_API_KEY,
#     api_version    = AZURE_API_VERSION,
# )
# print("✅ Azure OpenAI client initialised successfully!")

✅ Azure OpenAI client initialised successfully!


## Part 2: Build the Tax Knowledge Base

This is a **hardcoded knowledge base** — a Python dictionary representing common tax FAQs.

In Day 4 (RAG), you will replace this with real documents stored in a vector database.
For today, this is our "grounded" source of truth.

Alternatively, you can uncomment and run the following section to use .env variables.

In [5]:
# ── TAX KNOWLEDGE BASE ────────────────────────────────────────────────────────
# Organised by topic. Each entry has a 'question' pattern and an 'answer'.
# The search function will scan these for keyword matches.

TAX_KB = {
    "gst_standard_rate": {
        "keywords": ["gst", "goods and services tax", "standard rate", "gst rate"],
        "answer": (
            "The standard GST rate in India is 18% for most services. "
            "Goods are taxed at 5%, 12%, 18%, or 28% depending on the category. "
            "Essential goods attract 0% or 5%. Luxury and sin goods attract 28%."
        )
    },
    "tds_contractors": {
        "keywords": ["tds", "contractor", "section 194c", "194-c", "contractor payment"],
        "answer": (
            "Under Section 194C, TDS on contractor payments is 1% for individuals/HUF "
            "and 2% for companies. The threshold is Rs 30,000 per payment or "
            "Rs 1,00,000 in aggregate per financial year."
        )
    },
    "corporate_tax_india": {
        "keywords": ["corporate tax", "company tax rate", "domestic company", "corporate income"],
        "answer": (
            "Domestic companies with turnover up to Rs 400 crore: 25% base rate. "
            "Other domestic companies: 30% base rate. New manufacturing companies "
            "(set up after Oct 2019): 15% concessional rate under Section 115BAB. "
            "All rates subject to surcharge and cess (effective rate ~25.17% for most)."
        )
    },
    "section_80c": {
        "keywords": ["80c", "section 80c", "deduction", "tax saving", "lic", "ppf", "elss"],
        "answer": (
            "Section 80C allows deductions up to Rs 1,50,000 per year. "
            "Eligible investments include: LIC premiums, PPF contributions, ELSS mutual funds, "
            "NSC, 5-year bank FDs, children's tuition fees, home loan principal repayment, "
            "and EPF contributions. Available under Old Tax Regime only."
        )
    },
    "transfer_pricing": {
        "keywords": ["transfer pricing", "arm's length", "international transaction", "atp", "apl"],
        "answer": (
            "Transfer pricing rules (Section 92-92F) require that international transactions "
            "between associated enterprises be conducted at arm's length price. "
            "Mandatory documentation includes a Master File (if consolidated group revenue "
            "exceeds Rs 500 crore) and Local File for each reported international transaction. "
            "Due date: typically one month before ITR filing (October 31 for most companies)."
        )
    },
    "gst_input_tax_credit": {
        "keywords": ["itc", "input tax credit", "gst credit", "claim credit"],
        "answer": (
            "Input Tax Credit (ITC) under GST allows businesses to deduct the tax paid on "
            "inputs from tax payable on output. Conditions: supplier must have filed returns, "
            "buyer must have the invoice, goods/services must be received, and GST must have "
            "been paid to the government. ITC is not available on personal consumption or blocked credits (Rule 17)."
        )
    },
    "capital_gains_ltcg": {
        "keywords": ["capital gains", "ltcg", "long term", "equity", "section 112a"],
        "answer": (
            "Long-term capital gains (LTCG) on listed equity shares and equity mutual funds "
            "exceeding Rs 1 lakh are taxed at 10% without indexation (Section 112A, post Budget 2018). "
            "Holding period for LTCG: >12 months for equity, >24 months for immovable property, "
            ">36 months for debt mutual funds."
        )
    },
    "advance_tax": {
        "keywords": ["advance tax", "installment", "quarterly tax", "self assessment"],
        "answer": (
            "Advance tax is payable if tax liability exceeds Rs 10,000 in a year. "
            "Installments: 15% by June 15, 45% by September 15, 75% by December 15, "
            "100% by March 15. Interest under Sections 234B and 234C applies for shortfall. "
            "Senior citizens (60+) not having business income are exempt from advance tax."
        )
    },
}

print(f"✅ Knowledge base loaded with {len(TAX_KB)} topics:")
for key in TAX_KB:
    print(f"  • {key}")

✅ Knowledge base loaded with 8 topics:
  • gst_standard_rate
  • tds_contractors
  • corporate_tax_india
  • section_80c
  • transfer_pricing
  • gst_input_tax_credit
  • capital_gains_ltcg
  • advance_tax


### Build the Search Tool Function

In [6]:
def search_tax_knowledge(query: str) -> str:
    """
    Search the tax knowledge base for entries matching the query.
    Returns the best matching answer, or a 'not found' message.
    """
    query_lower = query.lower()
    matches = []

    for topic, data in TAX_KB.items():
        score = sum(1 for kw in data["keywords"] if kw in query_lower)
        if score > 0:
            matches.append((score, topic, data["answer"]))

    if not matches:
        return (
            "No specific information found in the knowledge base for that query. "
            "Please advise the user to consult official tax documentation or a qualified tax advisor."
        )

    # Sort by relevance score — highest first
    matches.sort(reverse=True)
    best_score, best_topic, best_answer = matches[0]

    result = f"[Source: {best_topic}]\n{best_answer}"

    # Include secondary matches if found
    if len(matches) > 1:
        additional = [f"  • {t}: (score {s})" for s, t, _ in matches[1:3]]
        result += "\n\nRelated topics also found:\n" + "\n".join(additional)

    return result


def calculate_tax(amount: float, rate: float) -> str:
    """
    Calculate tax amount given a base amount and percentage rate.
    Returns formatted tax computation.
    """
    if amount < 0:
        return "Error: Amount cannot be negative."
    if not (0 <= rate <= 100):
        return "Error: Rate must be between 0 and 100."

    tax_amount = round(amount * rate / 100, 2)
    net_amount = round(amount + tax_amount, 2)

    return (
        f"Tax Calculation:\n"
        f"  Base amount : ₹{amount:,.2f}\n"
        f"  Tax rate    : {rate}%\n"
        f"  Tax amount  : ₹{tax_amount:,.2f}\n"
        f"  Total       : ₹{net_amount:,.2f}"
    )


# ── Quick smoke test ──────────────────────────────────────────────────────────
print("SEARCH TEST:")
print(search_tax_knowledge("what is the TDS rate for contractor"))
print()
print("CALCULATOR TEST:")
print(calculate_tax(100000, 18))

SEARCH TEST:
[Source: tds_contractors]
Under Section 194C, TDS on contractor payments is 1% for individuals/HUF and 2% for companies. The threshold is Rs 30,000 per payment or Rs 1,00,000 in aggregate per financial year.

CALCULATOR TEST:
Tax Calculation:
  Base amount : ₹100,000.00
  Tax rate    : 18%
  Tax amount  : ₹18,000.00
  Total       : ₹118,000.00


## Part 3: Define Tools Schema & Implement ReAct Loop

The LLM decides **when** to call a tool and **which arguments** to pass.
We define the tools as JSON schemas so the model understands what each tool does.

In [7]:
# ── TOOLS SCHEMA — passed to the LLM ─────────────────────────────────────────
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_tax_knowledge",
            "description": (
                "Search the internal tax knowledge base for information about "
                "Indian tax regulations, rates, sections, and compliance requirements. "
                "Use this before answering any tax-specific factual question."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The tax-related search query (e.g. 'TDS rate contractors', 'GST rate services')"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_tax",
            "description": (
                "Calculate the tax amount for a given base amount and tax rate percentage. "
                "Use when the user asks for a specific tax computation."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {
                        "type": "number",
                        "description": "The base amount in Indian Rupees (e.g. 100000)"
                    },
                    "rate": {
                        "type": "number",
                        "description": "The tax rate as a percentage (e.g. 18 for 18%)"
                    }
                },
                "required": ["amount", "rate"]
            }
        }
    }
]

# ── TOOL DISPATCHER — maps function name → Python function ───────────────────
TOOL_FUNCTIONS = {
    "search_tax_knowledge": search_tax_knowledge,
    "calculate_tax":        calculate_tax,
}

print("✅ Tools schema defined:")
for t in TOOLS:
    print(f"  • {t['function']['name']}: {t['function']['description'][:60]}...")

✅ Tools schema defined:
  • search_tax_knowledge: Search the internal tax knowledge base for information about...
  • calculate_tax: Calculate the tax amount for a given base amount and tax rat...


In [8]:
# ── SYSTEM PROMPT ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are TaxBot, an expert AI assistant for tax professionals at a Big4 accounting firm.

Your responsibilities:
1. Answer tax questions accurately by ALWAYS searching the knowledge base first.
2. Perform calculations when specific numbers are requested.
3. Cite the source of your information (e.g., Section number, regulation name).
4. If information is not in the knowledge base, clearly state that and recommend consulting official sources.
5. Be concise and professional — your audience are experienced tax professionals.

IMPORTANT RULES:
- Never guess a tax rate without first calling search_tax_knowledge.
- Never fabricate statutory references.
- For any computation involving rupee amounts, use the calculate_tax tool.
- Always qualify answers with "as per current regulations — please verify for latest updates."
"""

print("✅ System prompt set")
print(f"Length: {len(SYSTEM_PROMPT)} characters")

✅ System prompt set
Length: 823 characters


In [9]:
# ── ReAct AGENT CLASS ─────────────────────────────────────────────────────────
class TaxFAQAgent:
    """
    A ReAct (Reasoning + Acting) agent for tax FAQ queries.

    Loop:
      1. User sends a query
      2. LLM decides whether to call a tool or answer directly
      3. If tool_call → execute tool → add result → loop again
      4. If no tool_call → final answer returned
    """

    def __init__(self, client, deployment_name, max_iterations: int = 8, verbose: bool = True):
        self.client          = client
        self.deployment_name = deployment_name
        self.max_iterations  = max_iterations
        self.verbose         = verbose
        self.conversation_history = []   # Multi-turn memory

    def _log(self, label: str, content: str, color_code: str = ""):
        """Print coloured trace output so students can see the ReAct loop."""
        if self.verbose:
            colors = {"thought": "\033[94m", "action": "\033[93m",
                      "observation": "\033[92m", "answer": "\033[96m", "": ""}
            reset  = "\033[0m"
            print(f"{colors.get(color_code,'')}[{label}]{reset} {content}")

    def _execute_tool(self, tool_call) -> str:
        """Dispatch a tool call and return its string result."""
        func_name = tool_call.function.name
        try:
            args = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError:
            return f"Error: could not parse arguments for {func_name}"

        if func_name not in TOOL_FUNCTIONS:
            return f"Error: unknown tool '{func_name}'"

        self._log("ACTION", f"Calling tool: {func_name}({args})", "action")
        result = TOOL_FUNCTIONS[func_name](**args)
        self._log("OBSERVATION", result[:300], "observation")
        return result

    def chat(self, user_message: str) -> str:
        """Process one user turn through the ReAct loop."""
        self._log("USER", user_message)
        self.conversation_history.append({"role": "user", "content": user_message})

        for iteration in range(1, self.max_iterations + 1):
            self._log("THOUGHT", f"Iteration {iteration}/{self.max_iterations}", "thought")

            # Build messages: system prompt + conversation history
            messages = [{"role": "system", "content": SYSTEM_PROMPT}] + self.conversation_history

            response = self.client.chat.completions.create(
                model        = self.deployment_name,
                messages     = messages,
                tools        = TOOLS,
                tool_choice  = "auto",
                temperature  = 0.1,    # Low temperature for factual tax answers
                max_tokens   = 600,
            )

            message = response.choices[0].message

            # ── Case 1: Model wants to call tool(s) ──────────────────────────
            if message.tool_calls:
                # Append model's tool-call message to history
                self.conversation_history.append({
                    "role":       "assistant",
                    "content":    message.content or "",
                    "tool_calls": [
                        {
                            "id":       tc.id,
                            "type":     "function",
                            "function": {
                                "name":      tc.function.name,
                                "arguments": tc.function.arguments
                            }
                        }
                        for tc in message.tool_calls
                    ]
                })

                # Execute each tool and append results
                for tool_call in message.tool_calls:
                    tool_result = self._execute_tool(tool_call)
                    self.conversation_history.append({
                        "role":         "tool",
                        "tool_call_id": tool_call.id,
                        "content":      tool_result,
                    })

            # ── Case 2: Model gives final answer ─────────────────────────────
            else:
                final_answer = message.content or "(No response generated)"
                self.conversation_history.append({
                    "role":    "assistant",
                    "content": final_answer
                })
                self._log("ANSWER", final_answer, "answer")
                print()
                return final_answer

        # Safety exit — max iterations reached
        msg = "Agent reached maximum iteration limit. Please simplify your question."
        self.conversation_history.append({"role": "assistant", "content": msg})
        return msg

    def reset_memory(self):
        """Clear conversation history — start a fresh session."""
        self.conversation_history = []
        print("🔄 Conversation memory cleared.")

    def show_history(self):
        """Display the full conversation history."""
        print("\n" + "="*60)
        print("CONVERSATION HISTORY")
        print("="*60)
        for i, msg in enumerate(self.conversation_history, 1):
            role = msg.get("role", "unknown").upper()
            content = msg.get("content", "") or ""
            tool_calls = msg.get("tool_calls", [])
            if tool_calls:
                names = [tc["function"]["name"] for tc in tool_calls]
                print(f"[{i}] {role} → called tools: {names}")
            elif content:
                print(f"[{i}] {role}: {content[:150]}{'...' if len(content)>150 else ''}")
        print("="*60 + "\n")


# Instantiate the agent
agent = TaxFAQAgent(client=client, deployment_name=AZURE_DEPLOYMENT_NAME, verbose=True)
print("✅ TaxFAQAgent ready!")

✅ TaxFAQAgent ready!


## Part 4: Test the Agent

Run the cells below one by one. Watch the **reasoning trace** to see the ReAct loop in action:
- `[THOUGHT]` — agent decides what to do
- `[ACTION]` — tool is called with arguments
- `[OBSERVATION]` — tool returns result
- `[ANSWER]` — final response to user

In [10]:
# ── TEST 1: Basic factual query — agent should call search_tax_knowledge ──────
print("TEST 1: Basic factual query")
print("-" * 60)
agent.reset_memory()
response = agent.chat("What is the TDS rate for contractor payments?")

TEST 1: Basic factual query
------------------------------------------------------------
🔄 Conversation memory cleared.
[USER] What is the TDS rate for contractor payments?
[THOUGHT] Iteration 1/8
[ACTION] Calling tool: search_tax_knowledge({'query': 'TDS rate for contractor payments'})
[OBSERVATION] [Source: tds_contractors]
Under Section 194C, TDS on contractor payments is 1% for individuals/HUF and 2% for companies. The threshold is Rs 30,000 per payment or Rs 1,00,000 in aggregate per financial year.
[THOUGHT] Iteration 2/8
[ANSWER] As per Section 194C, the TDS rate for contractor payments is:

- 1% for individuals/HUF
- 2% for companies

This applies if the payment exceeds Rs 30,000 per transaction or Rs 1,00,000 in aggregate during the financial year. Please verify for the latest updates.



In [11]:
# ── TEST 2: Calculation query — agent should call calculate_tax ───────────────
print("TEST 2: Tax calculation")
print("-" * 60)
agent.reset_memory()
response = agent.chat("Calculate the GST on a service invoice of Rs 2,50,000 at 18%")

TEST 2: Tax calculation
------------------------------------------------------------
🔄 Conversation memory cleared.
[USER] Calculate the GST on a service invoice of Rs 2,50,000 at 18%
[THOUGHT] Iteration 1/8
[ACTION] Calling tool: calculate_tax({'amount': 250000, 'rate': 18})
[OBSERVATION] Tax Calculation:
  Base amount : ₹250,000.00
  Tax rate    : 18%
  Tax amount  : ₹45,000.00
  Total       : ₹295,000.00
[THOUGHT] Iteration 2/8
[ANSWER] The GST on a service invoice of ₹2,50,000 at 18% is ₹45,000. The total invoice amount, including GST, will be ₹2,95,000.



In [12]:
# ── TEST 3: Multi-turn conversation — memory test ─────────────────────────────
print("TEST 3: Multi-turn conversation")
print("-" * 60)
agent.reset_memory()

# First turn
agent.chat("What is Section 80C?")

# Second turn — agent should remember context
agent.chat("What is the maximum deduction amount?")

# Third turn — calculate based on prior context
agent.chat("If I invest Rs 1,50,000 and save at 30% tax slab, how much tax do I save?")

TEST 3: Multi-turn conversation
------------------------------------------------------------
🔄 Conversation memory cleared.
[USER] What is Section 80C?
[THOUGHT] Iteration 1/8
[ACTION] Calling tool: search_tax_knowledge({'query': 'Section 80C'})
[OBSERVATION] [Source: section_80c]
Section 80C allows deductions up to Rs 1,50,000 per year. Eligible investments include: LIC premiums, PPF contributions, ELSS mutual funds, NSC, 5-year bank FDs, children's tuition fees, home loan principal repayment, and EPF contributions. Available under Old Tax Regime only.
[THOUGHT] Iteration 2/8
[ANSWER] Section 80C of the Income Tax Act allows deductions of up to ₹1,50,000 per year for specified investments and expenses. Eligible items include:

- Life Insurance Premiums (LIC)
- Public Provident Fund (PPF) contributions
- Equity Linked Savings Schemes (ELSS) mutual funds
- National Savings Certificates (NSC)
- 5-year fixed deposits with banks
- Children's tuition fees
- Principal repayment of home loans

'If you invest ₹1,50,000 under Section 80C and are in the 30% tax slab, you save ₹45,000 in taxes, as per current regulations — please verify for the latest updates.'

In [13]:
# Show the full conversation history
agent.show_history()


CONVERSATION HISTORY
[1] USER: What is Section 80C?
[2] ASSISTANT → called tools: ['search_tax_knowledge']
[3] TOOL: [Source: section_80c]
Section 80C allows deductions up to Rs 1,50,000 per year. Eligible investments include: LIC premiums, PPF contributions, ELSS mu...
[4] ASSISTANT: Section 80C of the Income Tax Act allows deductions of up to ₹1,50,000 per year for specified investments and expenses. Eligible items include:

- Lif...
[5] USER: What is the maximum deduction amount?
[6] ASSISTANT: The maximum deduction amount under Section 80C is ₹1,50,000 per financial year, as per current regulations — please verify for the latest updates.
[7] USER: If I invest Rs 1,50,000 and save at 30% tax slab, how much tax do I save?
[8] ASSISTANT → called tools: ['calculate_tax']
[9] TOOL: Tax Calculation:
  Base amount : ₹150,000.00
  Tax rate    : 30%
  Tax amount  : ₹45,000.00
  Total       : ₹195,000.00
[10] ASSISTANT: If you invest ₹1,50,000 under Section 80C and are in the 30% tax slab, 

## Exercise 2: Extend the Knowledge Base (30 min)

Add **3 new entries** to `TAX_KB` and test them with the agent.

Suggestions:
1. Income tax slabs for FY 2025-26 (New Tax Regime)
2. Section 44AD — Presumptive taxation for businesses
3. GST registration threshold

### Instructions:
1. Copy the pattern from an existing `TAX_KB` entry
2. Add your entry with at least 3 keywords
3. Test with `agent.chat("your question about new topic")`
4. Verify the agent finds and uses your new entry

In [14]:
# ── EXERCISE 2: Add your entries here ────────────────────────────────────────

# EXAMPLE STRUCTURE — replace with your own:
new_entries = {
    "new_tax_regime_slabs": {
        "keywords": ["new tax regime", "new regime", "income tax slab", "fy 2025", "budget 2025"],
        "answer": (
            "New Tax Regime slabs for FY 2025-26: "
            "Up to Rs 3 lakh: Nil. Rs 3-7 lakh: 5%. Rs 7-10 lakh: 10%. "
            "Rs 10-12 lakh: 15%. Rs 12-15 lakh: 20%. Above Rs 15 lakh: 30%. "
            "Standard deduction of Rs 75,000 available. No deductions under Chapter VI-A."
        )
    },
    # TODO: Add your second entry here
    # "section_44ad": {
    #     "keywords": [...],
    #     "answer": "..."
    # },
}

# Add to knowledge base
TAX_KB.update(new_entries)
print(f"✅ Knowledge base now has {len(TAX_KB)} entries:")
for k in TAX_KB:
    print(f"  • {k}")

✅ Knowledge base now has 9 entries:
  • gst_standard_rate
  • tds_contractors
  • corporate_tax_india
  • section_80c
  • transfer_pricing
  • gst_input_tax_credit
  • capital_gains_ltcg
  • advance_tax
  • new_tax_regime_slabs


In [15]:
# Test your new entries
agent.reset_memory()
agent.chat("What are the income tax slabs under the new tax regime for 2025?")

🔄 Conversation memory cleared.
[USER] What are the income tax slabs under the new tax regime for 2025?
[THOUGHT] Iteration 1/8
[ACTION] Calling tool: search_tax_knowledge({'query': 'income tax slabs new tax regime 2025'})
[OBSERVATION] [Source: new_tax_regime_slabs]
New Tax Regime slabs for FY 2025-26: Up to Rs 3 lakh: Nil. Rs 3-7 lakh: 5%. Rs 7-10 lakh: 10%. Rs 10-12 lakh: 15%. Rs 12-15 lakh: 20%. Above Rs 15 lakh: 30%. Standard deduction of Rs 75,000 available. No deductions under Chapter VI-A.
[THOUGHT] Iteration 2/8
[ANSWER] As per current regulations for FY 2025-26 under the new tax regime:

- Income up to ₹3,00,000: Nil
- ₹3,00,001 to ₹7,00,000: 5%
- ₹7,00,001 to ₹10,00,000: 10%
- ₹10,00,001 to ₹12,00,000: 15%
- ₹12,00,001 to ₹15,00,000: 20%
- Above ₹15,00,000: 30%

A standard deduction of ₹75,000 is available, but no deductions under Chapter VI-A are permitted. Please verify for the latest updates.



'As per current regulations for FY 2025-26 under the new tax regime:\n\n- Income up to ₹3,00,000: Nil\n- ₹3,00,001 to ₹7,00,000: 5%\n- ₹7,00,001 to ₹10,00,000: 10%\n- ₹10,00,001 to ₹12,00,000: 15%\n- ₹12,00,001 to ₹15,00,000: 20%\n- Above ₹15,00,000: 30%\n\nA standard deduction of ₹75,000 is available, but no deductions under Chapter VI-A are permitted. Please verify for the latest updates.'

## Exercise 3: Advanced Challenge — Add a New Tool (30 min)

**For advanced participants:**

Add a third tool: `get_filing_deadline(tax_type: str) -> str`

Requirements:
1. Implement the Python function with a dictionary of deadlines
2. Add it to the `TOOLS` schema
3. Register it in `TOOL_FUNCTIONS`
4. Test: `agent.chat("When is the deadline for GST annual return filing?")`

Hint: Use the same pattern as `search_tax_knowledge`.

In [16]:
# ── EXERCISE 3: Implement filing deadline tool ────────────────────────────────

def get_filing_deadline(tax_type: str) -> str:
    """
    Returns filing deadlines for various Indian tax forms.
    TODO: Complete this function with real deadlines.
    """
    deadlines = {
        # Example structure — complete with real deadlines
        "gstr-1":   "11th of the following month (monthly filers); Q+1 month 13th for quarterly",
        "gstr-3b":  "20th of the following month",
        "itr":      "July 31 for individuals; October 31 for companies requiring audit",
        # TODO: Add more deadlines:
        # "tds-return": "...",
        # "gstr-9": "...",
        # "advance-tax-q1": "...",
    }

    tax_lower = tax_type.lower().strip()
    for key, deadline in deadlines.items():
        if key in tax_lower or tax_lower in key:
            return f"Filing deadline for {key.upper()}: {deadline}"

    return f"Deadline for '{tax_type}' not found. Check Income Tax / GST portal for latest dates."


# TODO: Add to TOOLS list and TOOL_FUNCTIONS dict
# TOOL_FUNCTIONS["get_filing_deadline"] = get_filing_deadline

# TODO: Add schema to TOOLS list:
# {
#     "type": "function",
#     "function": {
#         "name": "get_filing_deadline",
#         "description": "...",
#         "parameters": { ... }
#     }
# }

# Test the function directly first:
print(get_filing_deadline("GSTR-1"))
print(get_filing_deadline("ITR"))

Filing deadline for GSTR-1: 11th of the following month (monthly filers); Q+1 month 13th for quarterly
Filing deadline for ITR: July 31 for individuals; October 31 for companies requiring audit
